# Redrob Hackathon — Phase 2: Candidate Ranking
**Pure Python · CPU-only · No network · ≤ 5 min for 100K candidates**

Loads `jd_spec.json` produced by Phase 1, streams `candidates.jsonl.gz`,
applies every rule from the spec, outputs a validated top-100 CSV.

### What Phase 2 implements (all from `jd_spec_fixed.json`)
| Dimension | Weight | What's checked |
|---|---|---|
| `required_signals` | 0.35 | 4 signals: embeddings, vector DB, Python, eval frameworks |
| `experience_fit` | 0.25 | Years in 5–9 band, ideal 6–8; career quality curve |
| `bonus_signals` | 0.09 | 5 bonus signals: fine-tuning, L2R, HR-tech, distributed, OSS |
| `location_logistics` | 0.10 | Pune/Noida preferred; notice ≤ 30 days ideal |
| `availability_freshness` | 0.10 | open_to_work + last_active + response_rate + applications |
| `hireability_signals` | 0.08 | interview_completion + offer_acceptance (−1 = neutral) |
| `verification_signals` | 0.03 | email + phone + linkedin verified |

**Hard gates (instant 0):** pure_research_no_production · recent_langchain_only · no_recent_coding  
**Soft penalties (multipliers):** title_chaser · framework_enthusiast · pure_consulting · wrong_domain · closed_source  
**Honeypot detection (disqualification-safe):** skill_proficiency_duration_mismatch · experience_predates_company_founding

> 📌 **Scoring metric is NDCG-heavy at the top:**  
> `0.50 × NDCG@10 + 0.30 × NDCG@50 + 0.15 × MAP + 0.05 × P@10`  
> → Precision at rank 1–10 is the dominant objective. Every design choice below optimises for that.

## Cell 1 — Imports & Paths

In [1]:
import csv
import gzip
import json
import math
import os
import re
import sys
import time
from concurrent.futures import ProcessPoolExecutor, as_completed
from datetime import date, datetime
from pathlib import Path

# ── Paths ─────────────────────────────────────────────────────────────────────
# Adjusted output path for Colab environment
CANDIDATES_GZ   = Path("/content/candidates.jsonl.gz")
CANDIDATES_JSONL= Path("/content/candidates.jsonl")
JD_SPEC_PATH    = Path("/content/jd_spec_fixed.json")   # Phase 1 output
OUTPUT_CSV      = Path("/content/submission.csv")

TODAY = date(2026, 6, 21)   # reference date (competition date)

print("✅ Imports OK")
print(f"   Candidates (gz) : {CANDIDATES_GZ}")
print(f"   JD spec         : {JD_SPEC_PATH}")
print(f"   Output CSV      : {OUTPUT_CSV}")

# ── File-completeness sanity check ───────────────────────────────────────────
# PATCHED: earlier Colab runs silently scored a TRUNCATED candidates.jsonl
# (1,086 and 5,394 lines instead of the expected 100,000) -- the upload to
# /content/ didn't finish transferring before the scoring cell ran, but
# nothing failed loudly; the notebook happily scored the partial file and
# reported a confident-looking (but meaningless) top-100. /content/ is
# ephemeral local disk in Colab, not network storage, so a slow/interrupted
# upload leaves a partial file with no error. This check fails LOUDLY and
# immediately if the candidate file isn't actually complete, instead of
# letting a partial run proceed silently.
EXPECTED_CANDIDATE_COUNT = 100_000

def _resolve_candidates_path() -> Path:
    if CANDIDATES_JSONL.exists():
        path = CANDIDATES_JSONL
    elif CANDIDATES_GZ.exists():
        path = CANDIDATES_GZ
    else:
        raise FileNotFoundError(
            f"Neither {CANDIDATES_JSONL} nor {CANDIDATES_GZ} exists. "
            f"Upload candidates.jsonl(.gz) to /content/ before running this notebook."
        )
    opener = gzip.open if path.suffix == ".gz" else open
    with opener(path, "rt", encoding="utf-8") as f:
        n_lines = sum(1 for line in f if line.strip())
    if n_lines != EXPECTED_CANDIDATE_COUNT:
        raise ValueError(
            f"Expected {EXPECTED_CANDIDATE_COUNT:,} candidate lines in {path}, "
            f"found {n_lines:,}. This usually means the upload to /content/ "
            f"didn't finish (Colab's /content/ is ephemeral local disk, not "
            f"network storage -- a slow/interrupted upload leaves a silently "
            f"truncated file). Re-upload the full file and re-run this cell "
            f"before proceeding; do NOT continue with a partial file."
        )
    print(f"✅ File completeness check passed: {n_lines:,} candidate lines found at {path}")
    return path

CANDIDATES_PATH = _resolve_candidates_path()


✅ Imports OK
   Candidates (gz) : /content/candidates.jsonl.gz
   JD spec         : /content/jd_spec_fixed.json
   Output CSV      : /content/submission.csv
✅ File completeness check passed: 100,000 candidate lines found at /content/candidates.jsonl.gz


## Cell 2 — Load JD Spec

Loads the `jd_spec.json` produced by Phase 1 and builds all fast-lookup
structures needed by the scorer (lowercased keyword sets, compiled regexes,
company founding-year table, etc.).

In [2]:
with open(JD_SPEC_PATH, "r", encoding="utf-8") as f:
    SPEC = json.load(f)

# ── Experience band ───────────────────────────────────────────────────────────
EXP_BAND = SPEC["experience_band"]

# ── Required signals ──────────────────────────────────────────────────────────
REQUIRED_SIGNALS = SPEC["required_signals"]
# Pre-compile: {signal_id -> (weight, frozenset of lowercase keywords)}
REQ_LOOKUP = {
    s["id"]: (s["weight"], frozenset(k.lower() for k in s["keyword_hints"]))
    for s in REQUIRED_SIGNALS
}
REQ_MAX_WEIGHT = sum(s["weight"] for s in REQUIRED_SIGNALS)

# ── Bonus signals ─────────────────────────────────────────────────────────────
BONUS_SIGNALS = SPEC["bonus_signals"]
# Override open_source signal keywords — spec's "AI" and "ML" are too broad
# (nearly every candidate mentions AI/ML somewhere). Replace with OSS-specific evidence.
_OPEN_SOURCE_KWS_OVERRIDE = frozenset([
    "open-source", "open source", "github.com", "arxiv", "arxiv.org",
    "conference paper", "published", "publication", "hugging face",
    "kaggle competition", "kaggle rank", "blog post", "technical blog",
    "workshop", "open source contribution", "maintainer", "contributor",
    "pull request", "merged pr", "starred", "forked",
])
BONUS_LOOKUP = {}
for s in BONUS_SIGNALS:
    if s["id"] == "open_source":
        BONUS_LOOKUP[s["id"]] = (s["weight"], _OPEN_SOURCE_KWS_OVERRIDE)
    else:
        BONUS_LOOKUP[s["id"]] = (s["weight"], frozenset(k.lower() for k in s["keyword_hints"]))
BONUS_MAX_WEIGHT = sum(s["weight"] for s in BONUS_SIGNALS)

# ── Soft disqualifiers ────────────────────────────────────────────────────────
SOFT_DQ = {d["id"]: d for d in SPEC["soft_disqualifiers"]}

# Consulting firms for pure_consulting_background check
CONSULTING_FIRMS = frozenset({
    "tcs", "infosys", "wipro", "accenture", "cognizant", "capgemini",
    "tech mahindra", "hcl", "mphasis", "mindtree", "genpact ai", "genpact",
    "ibm", "ey", "deloitte", "kpmg", "pwc",
})

# CV/Speech/Robotics keywords for wrong_domain_expertise
CV_SPEECH_KEYWORDS = frozenset({
    "computer vision", "object detection", "image segmentation", "image classification",
    "speech recognition", "speech synthesis", "tts", "asr", "robotics",
    "autonomous driving", "lidar", "3d reconstruction",
})
NLP_IR_KEYWORDS = frozenset({
    "nlp", "natural language", "text", "retrieval", "search", "ranking",
    "information retrieval", "question answering", "semantic", "bert", "gpt",
    "llm", "language model", "embeddings", "vector", "rag",
})

# ── Location ──────────────────────────────────────────────────────────────────
LOC_PREFERRED  = frozenset(l.lower() for l in SPEC["location_scoring"]["preferred_locations"])
LOC_ACCEPTABLE = frozenset(l.lower() for l in SPEC["location_scoring"]["acceptable_locations"])
TIER1_CITIES   = frozenset({
    "pune", "noida", "hyderabad", "mumbai", "delhi", "bangalore", "bengaluru",
    "chennai", "gurgaon", "gurugram", "kolkata", "ahmedabad", "jaipur",
})

# ── Honeypot: company founding years ─────────────────────────────────────────
FOUNDING_YEARS = {
    k.lower(): v["year"]
    for k, v in SPEC["honeypot_detection"]["patterns"][1]["company_founding_years"].items()
}
FICTIONAL_COMPANIES = frozenset(
    c.lower() for c in SPEC["honeypot_detection"]["patterns"][1]["fictional_placeholder_companies"]
)

# ── Dimension weights (top-level) ─────────────────────────────────────────────
DIM_W = SPEC["scoring_dimension_weights"]

print("✅ JD spec loaded")
print(f"   Required signals : {list(REQ_LOOKUP.keys())}")
print(f"   Bonus signals    : {list(BONUS_LOOKUP.keys())}")
print(f"   REQ max weight   : {REQ_MAX_WEIGHT}")
print(f"   BONUS max weight : {BONUS_MAX_WEIGHT}")
print(f"   Dimension weights: {DIM_W}")

✅ JD spec loaded
   Required signals : ['production_embeddings', 'vector_db_hybrid_search', 'strong_python', 'evaluation_frameworks']
   Bonus signals    : ['llm_fine_tuning', 'learning_to_rank', 'hr_tech_marketplace', 'distributed_systems', 'open_source']
   REQ max weight   : 3.9
   BONUS max weight : 1.4
   Dimension weights: {'required_signals': 0.35, 'experience_fit': 0.25, 'bonus_signals': 0.09, 'location_logistics': 0.1, 'availability_freshness': 0.1, 'hireability_signals': 0.08, 'verification_signals': 0.03}


## Cell 3 — Scoring Functions

Each function returns a float in [0, 1] (or a boolean for gates).
No global state: all inputs come from the candidate record + spec constants
defined in Cell 2 — safe for multiprocessing.

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# HELPERS
# ─────────────────────────────────────────────────────────────────────────────

def _candidate_text_blob(c: dict) -> str:
    """
    Concatenate all free-text fields into one lowercase string for keyword search.
    Includes: profile summary, all career_history descriptions, skill names,
    certifications, and the profile headline.
    This is searched for BOTH signal detection and disqualifier checking.
    """
    parts = [
        c["profile"].get("summary", ""),
        c["profile"].get("headline", ""),
    ]
    for role in c.get("career_history", []):
        parts.append(role.get("description", ""))
        parts.append(role.get("title", ""))
    for s in c.get("skills", []):
        parts.append(s.get("name", ""))
    for cert in c.get("certifications", []):
        parts.append(cert.get("name", ""))
    return " ".join(parts).lower()


def _skill_names_lower(c: dict) -> frozenset:
    return frozenset(s["name"].lower() for s in c.get("skills", []))


def _career_descriptions_lower(c: dict) -> str:
    return " ".join(
        r.get("description", "") for r in c.get("career_history", [])
    ).lower()


# ─────────────────────────────────────────────────────────────────────────────
# HONEYPOT DETECTION  (hard gate → score = 0)
# ─────────────────────────────────────────────────────────────────────────────

def is_honeypot(c: dict) -> tuple:
    """
    Returns (is_honeypot: bool, reason: str).
    Pattern 1: skill proficiency='expert' with duration_months < 12, in many skills.
    Pattern 2: career start date predates company founding year by a
    meaningful margin (see MIN_PREDATES_GAP_YEARS below).
    """
    skills = c.get("skills", [])

    # Pattern 1: expert + low duration
    expert_low_dur = [
        s["name"] for s in skills
        if s.get("proficiency") == "expert" and s.get("duration_months", 0) < 12
    ]
    # JD example: 10 skills. Fire strongly at ≥3 (catches the pattern, avoids
    # penalising a candidate who has one genuinely-new expertise).
    if len(expert_low_dur) >= 3:
        return True, f"honeypot:skill_duration_mismatch({len(expert_low_dur)} expert skills <12mo)"

    # Pattern 2: start date predates company founding by a real margin.
    #
    # PATCHED: an earlier version flagged ANY gap > 0 years, which produced
    # 281 flags across the full 100K dataset -- over 3x the spec's stated
    # ~80 honeypots total across BOTH patterns combined. Scanning the
    # gap-size distribution showed why: 165 candidates have a 1-year gap and
    # 69 have a 2-year gap, overwhelmingly concentrated on CRED (founded
    # 2018) with candidates starting 2016-2017 -- far too common and too
    # small to be a deliberately planted trap; far more consistent with
    # synthetic-data date-generation noise (a company "founding" date that
    # doesn't account for stealth-mode pre-launch hiring, rounding, etc.).
    # The distribution falls off sharply above 2 years (21 at 3yr, only 5 at
    # 4-5yr). Manually inspecting those 5 cases confirmed they match the
    # JD's named honeypot example almost exactly: candidates claiming 4-6
    # years tenure at Krutrim or Sarvam AI (both founded 2023) -- and
    # notably all 5 carry genuinely strong-looking titles (Senior NLP
    # Engineer, ML Engineer, Recommendation Systems Engineer), exactly the
    # "perfect on paper except impossible" honeypot design the spec
    # describes. Threshold set at >=4 years to capture this real pattern
    # while excluding the noise.
    MIN_PREDATES_GAP_YEARS = 4
    for role in c.get("career_history", []):
        company_lower = role.get("company", "").lower()
        if company_lower in FICTIONAL_COMPANIES:
            continue
        if company_lower not in FOUNDING_YEARS:
            continue
        start_date_str = role.get("start_date", "")
        if not start_date_str:
            continue
        try:
            start_year = int(start_date_str[:4])
        except ValueError:
            continue
        founding_year = FOUNDING_YEARS[company_lower]
        if founding_year - start_year >= MIN_PREDATES_GAP_YEARS:
            return True, (
                f"honeypot:predates_founding({role['company']} founded {founding_year}, "
                f"candidate start {start_year})"
            )

    return False, ""


# ─────────────────────────────────────────────────────────────────────────────
# HARD DISQUALIFIERS (hard gate → score = 0.001, buried at bottom)
# ─────────────────────────────────────────────────────────────────────────────

def check_hard_disqualifiers(c: dict, text_blob: str) -> tuple:
    """
    Returns (disqualified: bool, reason: str).
    Checks all three hard disqualifiers from the spec.
    """
    career = c.get("career_history", [])
    rs     = c.get("redrob_signals", {})
    skills = _skill_names_lower(c)

    # ── 1. pure_research_no_production ────────────────────────────────────────
    # Heuristic: ALL roles have research/academic titles AND none of the
    # descriptions mention production, deployed, shipped, users, inference.
    research_title_kws = frozenset(["researcher", "research engineer", "research scientist",
                                     "phd", "postdoc", "professor", "lecturer", "intern"])
    production_kws     = frozenset(["production", "deployed", "shipped", "users",
                                     "inference", "serving", "api", "real-time", "latency",
                                     "pipeline", "launch", "release"])
    if career:
        all_research_titles = all(
            any(kw in role["title"].lower() for kw in research_title_kws)
            for role in career
        )
        has_production_evidence = any(kw in text_blob for kw in production_kws)
        if all_research_titles and not has_production_evidence:
            return True, "hard_dq:pure_research_no_production"

    # ── 2. recent_langchain_only ──────────────────────────────────────────────
    # Disqualify if (a) GenAI-tool language (LangChain/RAG/vector-DB tools)
    # appears anywhere in the profile, (b) the candidate's actual
    # career_history descriptions show NO pre-LLM-era ML/retrieval/ranking
    # production work, and (c) the current title isn't already a clearly
    # ML/AI-track title.
    #
    # PATCHED: the original version checked has_pre_llm_experience against
    # text_blob, which includes skill NAMES mixed in alongside real narrated
    # career text -- so a candidate's own self-reported (unverified)
    # skills[] entries (e.g. "Recommendation Systems") could satisfy the
    # "pre-LLM-era ML evidence" exception with zero real corroboration. It
    # also used max() over self-reported AI-skill durations to judge
    # "recent", which let one inflated long-duration skill mask several
    # genuinely-recent ones. Concretely, CAND_0000021 (Project Manager,
    # summary explicitly says "taking online courses on RAG and vector
    # databases, experimenting with LangChain", career history is pure
    # team-management/stakeholder work) was NOT being disqualified despite
    # being the JD's own named example case, because her skills[] list
    # (likely inflated, like the dataset's other keyword-stuffer traps)
    # included "Recommendation Systems" and a Pinecone entry with 16mo
    # duration that masked her actual 5-7mo LangChain/Prompt-Engineering
    # entries. Fixed by checking ONLY _career_descriptions_lower (narrated,
    # role-grounded text) for pre-LLM evidence, and dropping the
    # duration_months dependency entirely -- presence of GenAI-tool language
    # anywhere (skills[] or free text) is itself sufficient signal once
    # there's no narrated counter-evidence.
    career_desc_only = _career_descriptions_lower(c)
    genai_tool_kws = frozenset(["langchain", "llamaindex", "openai api", "chatgpt api",
                                  "rag", "vector database", "pinecone", "weaviate", "chroma",
                                  "prompt engineering"])
    has_genai_tool_signal = bool(skills & genai_tool_kws) or any(kw in text_blob for kw in genai_tool_kws)
    pre_llm_kws = frozenset(["sklearn", "scikit-learn", "xgboost", "lightgbm",
                               "random forest", "gradient boost", "svm",
                               "recommendation", "ranking", "search", "retrieval",
                               "faiss", "annoy", "nmslib", "embedding model",
                               "sentence-transformers", "bert", "word2vec", "fasttext"])
    has_pre_llm_experience = any(kw in career_desc_only for kw in pre_llm_kws)
    ml_title_kws = frozenset(["machine learning", "ml engineer", "ai engineer", "data scientist",
                                "nlp", "deep learning", "computer vision", "research engineer",
                                "applied scientist", "recommendation systems engineer",
                                "search engineer", "applied ml engineer", "ml researcher",
                                "ai researcher", "applied ai"])
    current_title_lower = career[0]["title"].lower() if career else ""
    has_ml_title = any(kw in current_title_lower for kw in ml_title_kws)

    # NOTE: recent_langchain_only used to live here as a hard disqualifier.
    # PATCHED: after fixing it to check narrated career text only (instead
    # of the skill-polluted text_blob) and removing the unreliable
    # duration_months dependency, it correctly started firing on its real
    # target pattern -- but that pattern turns out to be ~32% of the full
    # 100K dataset (a large templated cohort: "Software engineer... I've
    # been keeping up with AI/ML at a self-learner level..."). Zeroing out
    # a third of the candidate pool via a single hard gate is too aggressive
    # and hard to defend if questioned -- these candidates should rank very
    # low, not be treated identically to a honeypot or a research-only
    # profile with zero production experience. Moved to
    # soft_disqualifier_multiplier as a steep (0.88) penalty instead -- see
    # that function below.

    # ── 3. no_recent_coding ───────────────────────────────────────────────────
    # GitHub activity score == 0 AND current/most-recent role has architecture/
    # management title AND description has no recent coding evidence.
    arch_mgmt_kws  = frozenset(["architect", "head of", "vp of", "director",
                                  "chief", "cto", "tech lead"])
    coding_kws     = frozenset(["implemented", "built", "wrote", "coded", "developed",
                                  "shipped", "deployed", "refactored", "optimised",
                                  "optimized", "pull request", "pr ", "commit",
                                  "python", "pytorch", "tensorflow"])
    github_score   = rs.get("github_activity_score", -1)

    if github_score == 0 and career:
        current_title_lower = career[0]["title"].lower()   # most recent first
        is_arch_mgmt = any(kw in current_title_lower for kw in arch_mgmt_kws)
        if is_arch_mgmt:
            # Check recent descriptions (last 2 roles)
            recent_desc = " ".join(
                r.get("description", "") for r in career[:2]
            ).lower()
            has_coding = any(kw in recent_desc for kw in coding_kws)
            if not has_coding:
                return True, "hard_dq:no_recent_coding(arch title + 0 github + no coding in recent roles)"

    return False, ""


# ─────────────────────────────────────────────────────────────────────────────
# SOFT DISQUALIFIER MULTIPLIERS (0.0 – 1.0, multiplicative)
# ─────────────────────────────────────────────────────────────────────────────

def soft_disqualifier_multiplier(c: dict, text_blob: str) -> tuple:
    """
    Returns (multiplier: float, flags: list[str]).
    Multiplier starts at 1.0 and is reduced by each soft DQ that fires.
    Multiple soft DQs stack multiplicatively.
    """
    multiplier = 1.0
    flags = []
    career = c.get("career_history", [])
    rs     = c.get("redrob_signals", {})
    skills = _skill_names_lower(c)

    # ── title_chaser (penalty = 0.8 → multiplier reduces by 0.8 → ×0.8) ──────
    # Fires if avg role duration < 18 months AND > 2 roles (excluding current)
    past_roles = [r for r in career if not r.get("is_current", False)]
    if len(past_roles) >= 2:
        avg_past_dur = sum(r["duration_months"] for r in past_roles) / len(past_roles)
        # Exception: current role duration ≥ 36 months signals intent to stay
        current_dur = career[0]["duration_months"] if career and career[0].get("is_current") else 0
        if avg_past_dur < 18 and current_dur < 36:
            multiplier *= (1.0 - SOFT_DQ["title_chaser"]["penalty_weight"])
            flags.append(f"soft_dq:title_chaser(avg_past_dur={avg_past_dur:.0f}mo)")

    # ── recent_langchain_only (penalty = 0.88, steep but not zeroing) ────────
    # Moved here from check_hard_disqualifiers -- see the NOTE in that
    # function for why a hard gate was too aggressive (fires on ~32% of the
    # full dataset, a large templated "self-learner level AI/ML dabbler"
    # cohort). As a steep soft penalty these candidates still rank far below
    # genuine fits (combines multiplicatively with career_relevance, which
    # independently also discounts non-ML-track careers) without being
    # treated identically to honeypots/hard-disqualified profiles.
    career_desc_only = _career_descriptions_lower(c)
    genai_tool_kws = frozenset(["langchain", "llamaindex", "openai api", "chatgpt api",
                                  "rag", "vector database", "pinecone", "weaviate", "chroma",
                                  "prompt engineering"])
    has_genai_tool_signal = bool(skills & genai_tool_kws) or any(kw in text_blob for kw in genai_tool_kws)
    pre_llm_kws = frozenset(["sklearn", "scikit-learn", "xgboost", "lightgbm",
                               "random forest", "gradient boost", "svm",
                               "recommendation", "ranking", "search", "retrieval",
                               "faiss", "annoy", "nmslib", "embedding model",
                               "sentence-transformers", "bert", "word2vec", "fasttext"])
    has_pre_llm_experience = any(kw in career_desc_only for kw in pre_llm_kws)
    ml_title_kws = frozenset(["machine learning", "ml engineer", "ai engineer", "data scientist",
                                "nlp", "deep learning", "computer vision", "research engineer",
                                "applied scientist", "recommendation systems engineer",
                                "search engineer", "applied ml engineer", "ml researcher",
                                "ai researcher", "applied ai"])
    current_title_lower = career[0]["title"].lower() if career else ""
    has_ml_title = any(kw in current_title_lower for kw in ml_title_kws)

    if has_genai_tool_signal and not has_pre_llm_experience and not has_ml_title:
        multiplier *= 0.12  # steep: pushes these well below genuine fits, not zero
        flags.append("soft_dq:recent_langchain_only(genai-tool signal, no narrated pre-LLM-era ML evidence)")

    # ── framework_enthusiast (penalty = 0.3) ──────────────────────────────────
    # Low github score + no systems-architecture evidence in descriptions
    github_score = rs.get("github_activity_score", -1)
    systems_kws  = frozenset(["architecture", "system design", "infrastructure",
                               "latency", "throughput", "distributed", "scale",
                               "indexing", "serving", "pipeline", "optimis",
                               "optimiz", "benchmark"])
    has_systems_evidence = any(kw in text_blob for kw in systems_kws)
    tutorial_kws = frozenset(["langchain", "llamaindex", "tutorial", "demo",
                               "prototype", "poc", "openai api", "chatgpt api"])
    has_tutorial_signal = any(kw in text_blob for kw in tutorial_kws)

    if has_tutorial_signal and not has_systems_evidence and (github_score < 20 or github_score == -1):
        multiplier *= (1.0 - SOFT_DQ["framework_enthusiast"]["penalty_weight"])
        flags.append("soft_dq:framework_enthusiast")

    # ── pure_consulting_background (penalty = 0.9) ────────────────────────────
    # ALL career roles are at consulting firms → no product experience
    if career:
        all_consulting = all(
            any(firm in r.get("company", "").lower() for firm in CONSULTING_FIRMS)
            or r.get("industry", "").lower() in {"it services", "consulting", "outsourcing"}
            for r in career
        )
        has_product_co = any(
            r.get("industry", "").lower() in {
                "software", "fintech", "e-commerce", "edtech", "healthtech",
                "saas", "internet", "ai", "product", "startup",
            }
            for r in career
        )
        if all_consulting and not has_product_co:
            multiplier *= (1.0 - SOFT_DQ["pure_consulting_background"]["penalty_weight"])
            flags.append("soft_dq:pure_consulting")

    # ── wrong_domain_expertise (penalty = 0.35) ───────────────────────────────
    # CV/Speech/Robotics dominant + no NLP/IR exposure
    cv_speech_skills = [s for s in c.get("skills", [])
                        if any(kw in s["name"].lower() for kw in CV_SPEECH_KEYWORDS)]
    nlp_ir_in_text   = any(kw in text_blob for kw in NLP_IR_KEYWORDS)

    if len(cv_speech_skills) >= 3 and not nlp_ir_in_text:
        multiplier *= (1.0 - SOFT_DQ["wrong_domain_expertise"]["penalty_weight"])
        flags.append(f"soft_dq:wrong_domain({len(cv_speech_skills)} CV/speech skills, no NLP/IR)")

    # ── closed_source_only (penalty = 0.25) ───────────────────────────────────
    # No OSS/publication evidence across all career descriptions
    open_kws = frozenset(["open-source", "open source", "github", "arxiv",
                           "paper", "conference", "publication", "published",
                           "hugging face", "kaggle", "blog", "talk", "workshop"])
    has_open = any(kw in text_blob for kw in open_kws)
    if not has_open and c["profile"]["years_of_experience"] >= 5:
        multiplier *= (1.0 - SOFT_DQ["closed_source_only"]["penalty_weight"])
        flags.append("soft_dq:closed_source_only")

    return round(multiplier, 4), flags


# ─────────────────────────────────────────────────────────────────────────────
# DIMENSION SCORERS (each returns 0.0 – 1.0)
# ─────────────────────────────────────────────────────────────────────────────

def score_required_signals(text_blob: str, skills_lower: frozenset,
                            career_history: list) -> tuple:
    """
    Upgraded signal matching v2: requires career-description corroboration
    for full credit. Skills-only matches (no career evidence) get 0.45x weight.
    This kills the Accountant-with-AI-skills problem.

    Scoring per signal:
    - career_hit (keyword in role descriptions/titles): full weight
    - skill_hit only (declared but no career evidence): 0.45 × weight
    - text_hits >= 2 (summary/headline mentions only): 0.25 × weight
    - no hit: 0
    """
    found = {}
    earned = 0.0

    career_blob = " ".join(
        r.get("description", "") + " " + r.get("title", "")
        for r in career_history
    ).lower()

    # Augment career_blob with proxy terms that imply Python usage
    PYTHON_PROXIES = frozenset([
        "pytorch", "tensorflow", "keras", "pyspark", "sklearn", "scikit",
        "fastapi", "flask", "django", "numpy", "pandas", "huggingface",
        "transformers", "langchain", "llamaindex", "jupyter",
    ])
    if any(prx in career_blob for prx in PYTHON_PROXIES):
        career_blob = career_blob + " python production code"

    # Augment career_blob with evaluation proxy terms.
    # Uses multi-word / specific phrases only to avoid substring false positives
    # (e.g. "roc" inside "processing", "metric" inside "symmetric").
    # Requires >= 2 hits before augmenting — one ambiguous hit is noise.
    EVAL_PROXIES = frozenset([
        "a/b test", "ab test", "split test", "online experiment",
        "click-through rate", "click through rate",
        "conversion rate", "conversion metric",
        "precision@", "recall@", "hit rate",
        "statistical significance", "hypothesis test",
        "offline eval", "online eval", "offline evaluation", "online evaluation",
        "user engagement metric", "engagement metric",
        "ranking metric", "search metric", "retrieval metric",
        "mean average precision", "mean reciprocal rank",
        "normalized discounted", "cumulative gain",
    ])
    eval_hits = sum(1 for prx in EVAL_PROXIES if prx in career_blob)
    if eval_hits >= 2:
        career_blob = career_blob + " a/b testing evaluation framework ranking metrics"
    elif eval_hits == 1:
        # Single weak signal: augment softly (won't trigger full credit alone)
        career_blob = career_blob + " ranking metrics"

    for sig_id, (weight, kw_set) in REQ_LOOKUP.items():
        skill_hit  = bool(skills_lower & kw_set)
        career_hit = any(kw in career_blob for kw in kw_set)
        text_hits  = sum(1 for kw in kw_set if kw in text_blob)

        if career_hit:
            credit = weight          # full credit — evidence in actual work
            found[sig_id] = True
        elif skill_hit:
            credit = weight * 0.45   # declared but not evidenced by career history
            found[sig_id] = "skill_only"
        elif text_hits >= 2:
            credit = weight * 0.25   # generic mention in summary/headline only
            found[sig_id] = "text_only"
        else:
            credit = 0.0
            found[sig_id] = False

        earned += credit

    return round(earned / REQ_MAX_WEIGHT, 4), found


def score_bonus_signals(text_blob: str, skills_lower: frozenset, career_history: list) -> tuple:
    """
    PATCHED: was never upgraded alongside score_required_signals v2 and
    still gave a bare skill_hit (any skills[]-name overlap, zero
    corroboration) full credit -- identical weight to genuine demonstrated
    experience. Brought up to parity: career_hit (keyword in actual
    role descriptions/titles) = full weight; skill_hit only = 0.45x;
    text_hits>=2 (summary/headline only) = 0.25x.
    """
    found = {}
    earned = 0.0

    career_blob = " ".join(
        r.get("description", "") + " " + r.get("title", "")
        for r in career_history
    ).lower()

    for sig_id, (weight, kw_set) in BONUS_LOOKUP.items():
        skill_hit  = bool(skills_lower & kw_set)
        career_hit = any(kw in career_blob for kw in kw_set)
        text_hits  = sum(1 for kw in kw_set if kw in text_blob)

        if career_hit:
            credit = weight
            found[sig_id] = True
        elif skill_hit:
            credit = weight * 0.45
            found[sig_id] = "skill_only"
        elif text_hits >= 2:
            credit = weight * 0.25
            found[sig_id] = "text_only"
        else:
            credit = 0.0
            found[sig_id] = False

        earned += credit

    return round(earned / BONUS_MAX_WEIGHT, 4), found



def _career_relevance_multiplier(c: dict) -> float:
    """
    Multiplier [0.20 – 1.0] based on how much of the career history is
    in AI/ML/software-adjacent roles vs. entirely irrelevant domains.

    Fixes the Accountant-at-rank-2 problem: an 8.7yr Accountant with
    AI keywords in skills should score near 0, not rank in the top 10.

    Returns 1.0 for candidates with relevant careers (no penalty).
    """
    career = c.get("career_history", [])
    if not career:
        return 1.0

    IRRELEVANT_KWS = frozenset([
        "accountant", "accounting", "auditor", "finance manager",
        "hr manager", "human resource", "talent acquisition", "recruiter",
        "civil engineer", "mechanical engineer", "structural engineer",
        "operations manager", "operations executive",
        "marketing manager", "marketing executive", "seo specialist",
        "content writer", "graphic designer", "illustrator",
        "customer support", "customer service", "call centre",
        "business analyst",
        "sales manager", "sales executive", "account executive",
        "legal", "lawyer", "advocate", "teacher", "professor",
    ])
    RELEVANT_KWS = frozenset([
        "machine learning", "ml engineer", "ai engineer", "data scientist",
        "nlp", "deep learning", "computer vision", "research engineer",
        "applied scientist", "data engineer", "software engineer",
        "backend engineer", "platform engineer", "mlops", "devops",
        "full stack", "python developer", "recommendation", "search engineer",
        "ranking", "retrieval", "llm", "generative", "applied ai",
        "analytics engineer", "ml researcher", "ai researcher",
    ])

    total_months     = sum(r["duration_months"] for r in career) or 1
    relevant_months  = 0
    irrelevant_months = 0

    for role in career:
        t   = role["title"].lower()
        dur = role["duration_months"]
        if any(kw in t for kw in RELEVANT_KWS):
            relevant_months += dur
        elif any(kw in t for kw in IRRELEVANT_KWS):
            irrelevant_months += dur

    rel_ratio  = relevant_months  / total_months
    irr_ratio  = irrelevant_months / total_months

    if rel_ratio >= 0.50:
        return 1.00    # majority of career is relevant
    elif rel_ratio >= 0.25:
        return 0.80    # mixed but meaningful relevant experience
    elif irr_ratio >= 0.75:
        return 0.15    # almost entirely irrelevant career — bury
    elif irr_ratio >= 0.50:
        return 0.40    # mostly irrelevant
    else:
        return 0.65    # unclear / recent pivotter


def score_experience_fit(c: dict) -> float:
    """
    Score based on years_of_experience against the spec's experience band.
    Curve: ramps up, peaks at ideal_min-ideal_max, gentle decay beyond.
    Also checks career quality: product-company experience gets a bonus.
    """
    yoe     = c["profile"]["years_of_experience"]
    mn      = EXP_BAND["min_years"]        # 5
    mx      = EXP_BAND["max_years"]        # 9
    id_min  = EXP_BAND["ideal_min_years"]  # 6
    id_max  = EXP_BAND["ideal_max_years"]  # 8

    # Band fit (0 – 1)
    if yoe < mn - 1:
        band_score = max(0.0, (yoe / mn) * 0.4)     # steep ramp below min
    elif yoe < mn:
        band_score = 0.40 + 0.20 * (yoe - (mn-1))   # 0.40 → 0.60 approaching min
    elif yoe <= id_min:
        band_score = 0.60 + 0.40 * ((yoe - mn) / (id_min - mn))  # 0.60 → 1.0
    elif yoe <= id_max:
        band_score = 1.00                             # ideal zone
    elif yoe <= mx:
        band_score = 1.00 - 0.10 * ((yoe - id_max) / (mx - id_max))  # 1.0 → 0.90
    else:
        band_score = max(0.60, 0.90 - 0.05 * (yoe - mx))  # gradual decay

    # Career quality bonus (up to +0.10): spent time at product companies
    career = c.get("career_history", [])
    product_months = sum(
        r["duration_months"] for r in career
        if r.get("industry", "").lower() not in {
            "it services", "consulting", "outsourcing", "bpo"
        }
        and not any(firm in r.get("company", "").lower() for firm in CONSULTING_FIRMS)
    )
    total_months = sum(r["duration_months"] for r in career) or 1
    product_ratio = product_months / total_months
    quality_bonus = 0.10 * product_ratio

    return round(min(1.0, band_score + quality_bonus), 4)


def score_location_logistics(c: dict) -> float:
    """
    Combined location + notice period score.
    Sub-weights (from spec) used to blend within this dimension:
      - location: 0.15 (relative)
      - notice:   0.10 (relative)
    Normalised so result is 0–1.
    """
    p  = c["profile"]
    rs = c["redrob_signals"]

    # Location score
    loc = (p.get("location", "") + " " + p.get("country", "")).lower()
    if any(city in loc for city in LOC_PREFERRED):
        loc_score = 1.00
    elif any(city in loc for city in LOC_ACCEPTABLE) or any(city in loc for city in TIER1_CITIES):
        loc_score = 0.65
    elif p.get("country", "").lower() == "india":
        loc_score = 0.45
    elif rs.get("willing_to_relocate", False):
        loc_score = 0.35
    else:
        loc_score = 0.15   # international, no relocation signal

    # Notice period score
    notice = rs.get("notice_period_days", 90)
    if notice <= 30:
        notice_score = 1.00
    elif notice <= 60:
        notice_score = 0.60
    elif notice <= 90:
        notice_score = 0.35
    else:
        notice_score = 0.10   # > 90 days is a real hiring blocker at Series A

    # Blend (sub-weights: loc=0.15, notice=0.10; normalise to sum=1)
    w_loc    = 0.15 / (0.15 + 0.10)   # 0.60
    w_notice = 0.10 / (0.15 + 0.10)   # 0.40
    return round(w_loc * loc_score + w_notice * notice_score, 4)


def score_availability_freshness(c: dict) -> float:
    """
    Measures active-in-job-market signals.
    Fields: open_to_work_flag, last_active_date, recruiter_response_rate,
            applications_submitted_30d.
    """
    rs = c["redrob_signals"]

    # open_to_work
    open_score = 1.0 if rs.get("open_to_work_flag", False) else 0.3

    # Recency: days since last active
    last_active_str = rs.get("last_active_date", "2020-01-01")
    try:
        last_active = datetime.strptime(last_active_str, "%Y-%m-%d").date()
        days_inactive = (TODAY - last_active).days
    except ValueError:
        days_inactive = 999
    if days_inactive <= 7:
        recency_score = 1.00
    elif days_inactive <= 30:
        recency_score = 0.85
    elif days_inactive <= 60:
        recency_score = 0.60
    elif days_inactive <= 120:
        recency_score = 0.35
    else:
        recency_score = 0.05   # staleness trap — spec explicitly calls this out

    # Recruiter response rate
    resp_rate  = rs.get("recruiter_response_rate", 0.0)
    resp_score = min(1.0, resp_rate / 0.7)   # 70%+ response rate = full score

    # Applications in last 30d (active seeker signal)
    apps = rs.get("applications_submitted_30d", 0)
    if apps == 0:
        apps_score = 0.3   # passive candidate
    elif apps <= 3:
        apps_score = 0.7
    else:
        apps_score = 1.0

    # Blend with equal weighting (4 sub-signals)
    return round(0.30 * open_score + 0.35 * recency_score +
                 0.20 * resp_score + 0.15 * apps_score, 4)


def score_hireability(c: dict) -> float:
    """
    interview_completion_rate + offer_acceptance_rate.
    CRITICAL: offer_acceptance_rate == -1 means NO DATA → treat as neutral (0.5),
    NOT as 0. Per spec: 'Do not score -1 as if it were a 0% acceptance rate.'
    """
    rs = c["redrob_signals"]

    interview_rate = rs.get("interview_completion_rate", 0.5)
    offer_rate_raw = rs.get("offer_acceptance_rate", -1)

    # -1 sentinel → neutral 0.5
    offer_rate = 0.5 if offer_rate_raw == -1 else offer_rate_raw

    return round(0.55 * interview_rate + 0.45 * offer_rate, 4)


def score_verification(c: dict) -> float:
    """
    email + phone + linkedin. Simple 0/0.33/0.67/1.0 step function.
    """
    rs = c["redrob_signals"]
    verified = sum([
        int(rs.get("verified_email", False)),
        int(rs.get("verified_phone", False)),
        int(rs.get("linkedin_connected", False)),
    ])
    return round(verified / 3.0, 4)


print("✅ All scoring functions defined")
print("   Functions: is_honeypot | check_hard_disqualifiers | soft_disqualifier_multiplier")
print("              score_required_signals | score_bonus_signals | score_experience_fit")
print("              score_location_logistics | score_availability_freshness")
print("              score_hireability | score_verification")

✅ All scoring functions defined
   Functions: is_honeypot | check_hard_disqualifiers | soft_disqualifier_multiplier
              score_required_signals | score_bonus_signals | score_experience_fit
              score_location_logistics | score_availability_freshness
              score_hireability | score_verification


## Cell 4 — Composite Scorer & Reasoning Generator

Wires all dimension scorers into a single `score_candidate()` function
and builds per-candidate reasoning strings that satisfy the spec's
Stage 4 quality checks (no hallucinations, no templates, specific facts).

In [4]:
def score_candidate(c: dict) -> dict:
    """
    Full pipeline for one candidate. Returns a result dict containing:
    - composite: final score (0.0 – 1.0)
    - dim_scores: per-dimension breakdown
    - flags: list of honeypot / disqualifier flags
    - reasoning: string for the CSV
    """
    cid = c["candidate_id"]

    # ── Honeypot gate ─────────────────────────────────────────────────────────
    hp, hp_reason = is_honeypot(c)
    if hp:
        return {
            "candidate_id": cid,
            "composite": 0.0001,
            "dim_scores": {},
            "flags": [hp_reason],
            "reasoning": f"Excluded: profile integrity check failed ({hp_reason})",
        }

    # ── Pre-compute shared structures (avoids redundant string ops) ───────────
    text_blob    = _candidate_text_blob(c)
    skills_lower = _skill_names_lower(c)

    # ── Hard disqualifier gate ────────────────────────────────────────────────
    dq, dq_reason = check_hard_disqualifiers(c, text_blob)
    if dq:
        return {
            "candidate_id": cid,
            "composite": 0.0002,
            "dim_scores": {},
            "flags": [dq_reason],
            "reasoning": f"Disqualified: {dq_reason}",
        }

    # ── Soft disqualifier multiplier ─────────────────────────────────────────
    soft_mult, soft_flags = soft_disqualifier_multiplier(c, text_blob)

    # ── Dimension scores ─────────────────────────────────────────────────────
    career_rel   = _career_relevance_multiplier(c)
    req_score,   req_found   = score_required_signals(text_blob, skills_lower,
                                                        c.get("career_history", []))
    bonus_score, bonus_found = score_bonus_signals(text_blob, skills_lower,
                                                     c.get("career_history", []))
    exp_score                = score_experience_fit(c)
    loc_score                = score_location_logistics(c)
    avail_score              = score_availability_freshness(c)
    hire_score               = score_hireability(c)
    verif_score              = score_verification(c)

    # ── Composite (spec weights, then soft multiplier) ────────────────────────
    w = DIM_W
    base = (
        w["required_signals"]    * req_score   +
        w["experience_fit"]      * exp_score   +
        w["bonus_signals"]       * bonus_score +
        w["location_logistics"]  * loc_score   +
        w["availability_freshness"] * avail_score +
        w["hireability_signals"] * hire_score  +
        w["verification_signals"]* verif_score
    )
    # career_rel applied BEFORE soft_mult:
    # an irrelevant career can't be rescued by good platform signals
    composite = round(min(1.0, base * career_rel * soft_mult), 6)

    dim_scores = {
        "required_signals": req_score,
        "experience_fit":   exp_score,
        "bonus_signals":    bonus_score,
        "location_logistics": loc_score,
        "availability_freshness": avail_score,
        "hireability_signals": hire_score,
        "verification_signals": verif_score,
        "career_relevance": career_rel,
        "soft_mult": soft_mult,
        "req_found": req_found,
        "bonus_found": bonus_found,
    }

    # ── Reasoning string ──────────────────────────────────────────────────────
    reasoning = _build_reasoning(c, composite, dim_scores, req_found,
                                  bonus_found, soft_flags)

    return {
        "candidate_id": cid,
        "composite": composite,
        "dim_scores": dim_scores,
        "flags": soft_flags,
        "reasoning": reasoning,
    }


def _build_reasoning(c: dict, score: float, dims: dict,
                     req_found: dict, bonus_found: dict,
                     soft_flags: list) -> str:
    """
    Builds a specific, non-hallucinated, per-candidate reasoning string.
    Pulls only facts that are literally present in the candidate's record.

    Spec Stage 4 requirements:
    ✓ References specific facts (title, years, skills, signal values)
    ✓ Connects to specific JD requirements
    ✓ Acknowledges gaps honestly
    ✗ Never mentions employers/skills not in the profile
    ✗ Never contradicts the rank
    """
    p   = c["profile"]
    rs  = c["redrob_signals"]
    yoe = p["years_of_experience"]

    # ── Lead with strongest fit signal ────────────────────────────────────────
    found_req_ids = [k for k, v in req_found.items() if v]
    found_bon_ids = [k for k, v in bonus_found.items() if v]

    # Map signal IDs to human-readable JD requirements
    req_labels = {
        "production_embeddings": "embeddings/retrieval systems",
        "vector_db_hybrid_search": "vector DB / hybrid search",
        "strong_python": "strong Python",
        "evaluation_frameworks": "evaluation frameworks (NDCG/A-B testing)",
    }
    bon_labels = {
        "llm_fine_tuning": "LLM fine-tuning",
        "learning_to_rank": "learning-to-rank",
        "hr_tech_marketplace": "HR-tech/marketplace exposure",
        "distributed_systems": "distributed systems / inference optimization",
        "open_source": "open-source contributions",
    }

    # Current title and company (safe — always in profile)
    current_role = f"{p['current_title']} at {p['current_company']}"

    # Build parts
    parts = []

    # Part 1: who they are + required signal hits
    if found_req_ids:
        req_str = ", ".join(req_labels[r] for r in found_req_ids[:2])
        parts.append(
            f"{yoe:.1f}yr {p['current_title']} with evidence of {req_str}"
        )
    else:
        parts.append(
            f"{yoe:.1f}yr {p['current_title']}; no direct match on core required signals"
        )

    # Part 2: missing required signals (gaps)
    missing_req = [k for k, v in req_found.items() if not v]
    if missing_req:
        gap_str = ", ".join(req_labels[m] for m in missing_req[:2])
        parts.append(f"gap: {gap_str} not evidenced")

    # Part 3: bonus signals
    if found_bon_ids:
        bon_str = ", ".join(bon_labels[b] for b in found_bon_ids[:2])
        parts.append(f"bonus: {bon_str}")

    # Part 4: behavioural / availability note
    last_active = rs.get("last_active_date", "unknown")
    open_flag   = rs.get("open_to_work_flag", False)
    notice      = rs.get("notice_period_days", 90)
    if open_flag and notice <= 30:
        parts.append(f"open to work, {notice}d notice, active {last_active}")
    elif not open_flag:
        parts.append(f"not marked open to work (last active {last_active})")

    # Part 5: soft flags
    if soft_flags:
        flag_str = "; ".join(f.replace("soft_dq:", "") for f in soft_flags[:2])
        parts.append(f"concerns: {flag_str}")

    # Join and cap at 300 chars (fits cleanly in CSV)
    result = ". ".join(parts)
    return result[:300]


print("✅ Composite scorer and reasoning builder ready")

# Quick smoke-test on CAND_0000001 (Ira Vora — known profile)
_smoke = {
    "candidate_id": "CAND_0000001",
    "profile": {
        "anonymized_name": "Ira Vora",
        "headline": "Backend Engineer | SQL, Spark, Cloud",
        "summary": "Backend/data hybrid building ML-adjacent pipelines.",
        "location": "Toronto", "country": "Canada",
        "years_of_experience": 6.9,
        "current_title": "Backend Engineer",
        "current_company": "Mindtree",
        "current_company_size": "10001+",
        "current_industry": "IT Services",
    },
    "career_history": [
        {"company": "Mindtree", "title": "Backend Engineer",
         "start_date": "2024-03-08", "end_date": None,
         "duration_months": 27, "is_current": True,
         "industry": "IT Services", "company_size": "10001+",
         "description": "Kafka and Spark Streaming pipelines for user-activity platform. Adjacent ML exposure."},
        {"company": "Dunder Mifflin", "title": "Analytics Engineer",
         "start_date": "2019-07-03", "end_date": "2024-01-08",
         "duration_months": 55, "is_current": False,
         "industry": "Paper Products", "company_size": "201-500",
         "description": "Apache Airflow, PySpark, Snowflake, dbt data pipelines."},
    ],
    "education": [{"institution": "LPU", "degree": "B.E.",
                   "field_of_study": "Computer Science",
                   "start_year": 2017, "end_year": 2020,
                   "grade": "8.24 CGPA", "tier": "tier_3"}],
    "skills": [
        {"name": "NLP", "proficiency": "advanced", "endorsements": 37, "duration_months": 26},
        {"name": "Fine-tuning LLMs", "proficiency": "advanced", "endorsements": 21, "duration_months": 36},
        {"name": "Milvus", "proficiency": "advanced", "endorsements": 40, "duration_months": 35},
        {"name": "Weights & Biases", "proficiency": "intermediate", "endorsements": 13, "duration_months": 30},
        {"name": "Kafka", "proficiency": "intermediate", "endorsements": 4, "duration_months": 9},
        {"name": "AWS", "proficiency": "beginner", "endorsements": 5, "duration_months": 8},
        {"name": "Flask", "proficiency": "beginner", "endorsements": 15, "duration_months": 15},
    ],
    "certifications": [],
    "languages": [{"language": "English", "proficiency": "professional"}],
    "redrob_signals": {
        "profile_completeness_score": 86.9,
        "signup_date": "2025-10-16",
        "last_active_date": "2026-05-20",
        "open_to_work_flag": True,
        "profile_views_received_30d": 23,
        "applications_submitted_30d": 2,
        "recruiter_response_rate": 0.34,
        "avg_response_time_hours": 177.8,
        "skill_assessment_scores": {"NLP": 38.8, "Fine-tuning LLMs": 41.6},
        "connection_count": 356,
        "endorsements_received": 35,
        "notice_period_days": 60,
        "expected_salary_range_inr_lpa": {"min": 18.7, "max": 36.1},
        "preferred_work_mode": "onsite",
        "willing_to_relocate": False,
        "github_activity_score": 9.2,
        "search_appearance_30d": 249,
        "saved_by_recruiters_30d": 4,
        "interview_completion_rate": 0.71,
        "offer_acceptance_rate": 0.58,
        "verified_email": True,
        "verified_phone": True,
        "linkedin_connected": False,
    },
}
_r = score_candidate(_smoke)
print(f"\nSmoke test CAND_0000001:")
print(f"  composite = {_r['composite']}")
ds = _r['dim_scores']
print(f"  composite      = {_r['composite']}")
print(f"  career_rel     = {ds.get('career_relevance','?')}")
print(f"  soft_mult      = {ds.get('soft_mult','?')}")
print(f"  required_sigs  = {ds.get('required_signals','?')}")
print(f"  req_found      = {ds.get('req_found','?')}")
print(f"  flags          = {_r['flags']}")
print(f"  flags = {_r['flags']}")
print(f"  reasoning = {_r['reasoning']}")

✅ Composite scorer and reasoning builder ready

Smoke test CAND_0000001:
  composite = 0.415445
  composite      = 0.415445
  career_rel     = 1.0
  soft_mult      = 0.75
  required_sigs  = 0.3462
  req_found      = {'production_embeddings': False, 'vector_db_hybrid_search': 'skill_only', 'strong_python': True, 'evaluation_frameworks': False}
  flags          = ['soft_dq:closed_source_only']
  flags = ['soft_dq:closed_source_only']
  reasoning = 6.9yr Backend Engineer with evidence of vector DB / hybrid search, strong Python. gap: embeddings/retrieval systems, evaluation frameworks (NDCG/A-B testing) not evidenced. bonus: LLM fine-tuning. concerns: closed_source_only


## Cell 5 — Stream & Score All 100K Candidates

Uses `concurrent.futures.ProcessPoolExecutor` with `chunksize=500` for
CPU parallelism. Falls back to single-threaded if multiprocessing fails
(Kaggle CPU-only notebooks sometimes restrict subprocess spawning).

**Memory:** streams JSONL line-by-line — never loads all 100K into RAM
simultaneously. Intermediate results are collected as `(score, id, reasoning)`
tuples only (< 50 MB for 100K candidates).

In [5]:
def _score_line(line: str) -> dict:
    """Top-level function for ProcessPoolExecutor (must be picklable)."""
    try:
        c = json.loads(line)
        return score_candidate(c)
    except json.JSONDecodeError as e:
        return {
            "candidate_id": "MALFORMED_JSON_LINE_ERROR",
            "composite": 0.0000001,
            "dim_scores": {},
            "flags": [f"json_decode_error:{e}"],
            "reasoning": f"Malformed JSON input: {e}",
        }

def load_candidate_lines():
    """Generator: yield one JSONL line at a time. Detects Gzip vs Plain text."""
    target_path = CANDIDATES_GZ if CANDIDATES_GZ.exists() else CANDIDATES_JSONL

    if not target_path.exists():
        raise FileNotFoundError(f"No candidate file found at {target_path}")

    # Check for Gzip magic number (0x1f 0x8b)
    with open(target_path, "rb") as f_test:
        is_gzip = f_test.read(2) == b'\x1f\x8b'

    opener = gzip.open(target_path, "rt", encoding="utf-8") if is_gzip else open(target_path, "r", encoding="utf-8")

    with opener as f:
        for line in f:
            line = line.strip()
            if line:
                yield line

def rank_candidates(max_workers: int = 4, chunk_size: int = 500):
    results = []
    n_total = 0
    n_honeypot = 0
    n_dq = 0
    n_json_errors = 0
    start = time.time()

    try:
        lines = list(load_candidate_lines())
    except Exception as e:
        print(f"❌ Failed to load lines: {e}")
        return []

    n_total = len(lines)
    print(f"Loaded {n_total:,} candidate lines. Starting scoring...")

    try:
        with ProcessPoolExecutor(max_workers=max_workers) as pool:
            for result in pool.map(_score_line, lines, chunksize=chunk_size):
                results.append(result)
                if result["composite"] == 0.0000001: n_json_errors += 1
                elif result["composite"] == 0.0001: n_honeypot += 1
                elif result["composite"] == 0.0002: n_dq += 1
    except Exception as e:
        print(f"Multiprocessing failed ({e}), falling back to serial...")
        results = [_score_line(line) for line in lines]

    results.sort(key=lambda x: (-x["composite"], x["candidate_id"]))

    elapsed = time.time() - start
    print(f"\n✅ Scored {n_total:,} candidates in {elapsed:.1f}s")
    return results

all_results = rank_candidates(max_workers=4, chunk_size=500)

Loaded 100,000 candidate lines. Starting scoring...

✅ Scored 100,000 candidates in 76.1s


## Cell 6 — Write Submission CSV

In [6]:
def write_submission_csv(results: list, output_path: Path, n: int = 100):
    """
    Write top-N to submission CSV.
    Guarantees: exactly 100 rows, ranks 1-100, non-increasing scores,
    no duplicate IDs, UTF-8, no BOM.
    """
    top = results[:n]
    output_path.parent.mkdir(parents=True, exist_ok=True)

    with open(output_path, "w", newline="", encoding="utf-8") as f:
        writer = csv.writer(f, quoting=csv.QUOTE_MINIMAL)
        writer.writerow(["candidate_id", "rank", "score", "reasoning"])

        prev_score = None
        for rank, r in enumerate(top, 1):
            score = r["composite"]
            # Enforce non-increasing (floating-point safety)
            if prev_score is not None:
                score = min(score, prev_score)
            prev_score = score

            reasoning = (r.get("reasoning") or "").replace("\n", " ").replace("\r", "").strip()
            reasoning = reasoning[:300]

            writer.writerow([
                r["candidate_id"],
                rank,
                f"{score:.6f}",
                reasoning,
            ])

    print(f"✅ Written {n} rows to {output_path}")
    print(f"   File size: {output_path.stat().st_size / 1024:.1f} KB")


write_submission_csv(all_results, OUTPUT_CSV)

✅ Written 100 rows to /content/submission.csv
   File size: 18.9 KB


## Cell 7 — Inline Validation

Replicates the checks from the official `validate_submission.py`.
Run this **before** uploading. Fix any errors before submitting.

In [7]:
def validate_submission(csv_path: Path) -> bool:
    errors = []

    with open(csv_path, "r", encoding="utf-8") as f:
        reader = csv.reader(f)
        header = next(reader)
        rows = [r for r in reader if any(c.strip() for c in r)]

    # Header check
    expected_header = ["candidate_id", "rank", "score", "reasoning"]
    if header != expected_header:
        errors.append(f"Header mismatch: got {header}, expected {expected_header}")

    # Row count
    if len(rows) != 100:
        errors.append(f"Expected 100 rows, got {len(rows)}")

    seen_ids   = set()
    seen_ranks = set()
    rows_by_rank = []

    for i, row in enumerate(rows, 2):
        if len(row) < 4:
            errors.append(f"Row {i}: too few columns ({len(row)})")
            continue
        cid, rank_s, score_s, reasoning = row[0], row[1], row[2], row[3]

        # candidate_id format
        if not re.match(r"^CAND_[0-9]{7}$", cid):
            errors.append(f"Row {i}: invalid candidate_id '{cid}'")
        if cid in seen_ids:
            errors.append(f"Row {i}: duplicate candidate_id '{cid}'")
        seen_ids.add(cid)

        # rank
        try:
            rank = int(rank_s)
        except ValueError:
            errors.append(f"Row {i}: rank not integer '{rank_s}'")
            continue
        if rank < 1 or rank > 100:
            errors.append(f"Row {i}: rank {rank} out of range 1-100")
        if rank in seen_ranks:
            errors.append(f"Row {i}: duplicate rank {rank}")
        seen_ranks.add(rank)

        # score
        try:
            score = float(score_s)
        except ValueError:
            errors.append(f"Row {i}: score not float '{score_s}'")
            score = 0.0
        rows_by_rank.append((rank, score))

        # reasoning quality checks
        if not reasoning.strip():
            errors.append(f"Row {i} ({cid}): empty reasoning")

    # Non-increasing scores
    rows_by_rank.sort(key=lambda x: x[0])
    for j in range(len(rows_by_rank) - 1):
        r1, s1 = rows_by_rank[j]
        r2, s2 = rows_by_rank[j + 1]
        if s1 < s2 - 1e-9:
            errors.append(f"Score not non-increasing: rank {r1} score {s1:.6f} < rank {r2} score {s2:.6f}")
            break

    # Missing ranks
    missing = sorted(set(range(1, 101)) - seen_ranks)
    if missing:
        errors.append(f"Missing ranks: {missing[:10]}")

    # Results
    if errors:
        print(f"❌ {len(errors)} validation error(s) found:")
        for e in errors:
            print(f"   • {e}")
        return False
    else:
        print("✅ All validation checks passed!")
        scores_sorted = [s for _, s in rows_by_rank]
        print(f"   Rows         : {len(rows)}")
        print(f"   Score range  : {scores_sorted[0]:.6f} → {scores_sorted[-1]:.6f}")
        print(f"   Unique IDs   : {len(seen_ids)}")
        return True


validate_submission(OUTPUT_CSV)

✅ All validation checks passed!
   Rows         : 100
   Score range  : 0.885821 → 0.743108
   Unique IDs   : 100


True

## Cell 8 — Honeypot Audit

Spec §7: honeypot rate > 10% in top-100 = Stage 3 disqualification.  
This cell verifies the constraint holds. If it fails, the scoring is broken.

In [8]:
print("=== HONEYPOT AUDIT — Top 100 ===")

hp_in_top100 = [
    r for r in all_results[:100]
    if any("honeypot" in f for f in (r.get("flags") or []))
]
hp_rate = len(hp_in_top100) / 100

print(f"  Honeypots in top-100 : {len(hp_in_top100)}/100 = {hp_rate:.1%}")
if hp_in_top100:
    for r in hp_in_top100[:5]:
        print(f"    {r['candidate_id']}  score={r['composite']:.4f}  {r['flags']}")

if hp_rate > 0.10:
    print("❌ ABOVE 10% — would be DISQUALIFIED at Stage 3!")
    print("   Check is_honeypot() logic and thresholds in Cell 3.")
else:
    print("✅ Below 10% threshold — safe.")

# Also show hard DQ rate in top-100 (should be 0 — DQ'd candidates score 0.0002)
dq_in_top100 = [
    r for r in all_results[:100]
    if any("hard_dq" in f for f in (r.get("flags") or []))
]
print(f"\n  Hard DQ in top-100  : {len(dq_in_top100)}/100")
if dq_in_top100:
    print("  ⚠️  Hard-DQ'd candidates should score 0.0002, not appear in top-100.")
    print("  Check score_candidate() gates in Cell 4.")

=== HONEYPOT AUDIT — Top 100 ===
  Honeypots in top-100 : 0/100 = 0.0%
✅ Below 10% threshold — safe.

  Hard DQ in top-100  : 0/100


## Cell 9 — Top-10 Deep-Dive

Given 80% of the score is NDCG@10, inspect the top-10 in detail.
Verify each is genuinely a strong AI/ML engineering candidate.

In [9]:
print("=== TOP-10 DETAILED BREAKDOWN ===")
print()

for rank, r in enumerate(all_results[:10], 1):
    cid  = r["candidate_id"]
    dims = r.get("dim_scores", {})
    print(f"{'─'*65}")
    print(f"  #{rank:2d}  {cid}   composite={r['composite']:.4f}")
    print(f"       Required={dims.get('required_signals','?'):.3f}  "
          f"ExpFit={dims.get('experience_fit','?'):.3f}  "
          f"Bonus={dims.get('bonus_signals','?'):.3f}  "
          f"Loc={dims.get('location_logistics','?'):.3f}  "
          f"Avail={dims.get('availability_freshness','?'):.3f}  "
          f"Hire={dims.get('hireability_signals','?'):.3f}  "
          f"Verif={dims.get('verification_signals','?'):.3f}  "
          f"SoftMult={dims.get('soft_mult','?'):.3f}")

    req_f = dims.get("req_found", {})
    bon_f = dims.get("bonus_found", {})
    print(f"       Req signals  : "
          f"{'✓' if req_f.get('production_embeddings') else '✗'} embeddings  "
          f"{'✓' if req_f.get('vector_db_hybrid_search') else '✗'} vectorDB  "
          f"{'✓' if req_f.get('strong_python') else '✗'} python  "
          f"{'✓' if req_f.get('evaluation_frameworks') else '✗'} eval")
    print(f"       Bonus signals: "
          f"{'✓' if bon_f.get('llm_fine_tuning') else '✗'} finetuning  "
          f"{'✓' if bon_f.get('learning_to_rank') else '✗'} L2R  "
          f"{'✓' if bon_f.get('hr_tech_marketplace') else '✗'} HR-tech  "
          f"{'✓' if bon_f.get('distributed_systems') else '✗'} distributed  "
          f"{'✓' if bon_f.get('open_source') else '✗'} OSS")
    if r.get("flags"):
        print(f"       Flags: {r['flags']}")
    print(f"       Reasoning: {r['reasoning'][:120]}...")
    print()

print(f"{'─'*65}")
print()
print("=== SCORE DISTRIBUTION ===")
thresholds = [1, 5, 10, 25, 50, 100, 500, 1000, 5000]
for t in thresholds:
    if t <= len(all_results):
        print(f"  Rank {t:5d}: score = {all_results[t-1]['composite']:.6f}")

=== TOP-10 DETAILED BREAKDOWN ===

─────────────────────────────────────────────────────────────────
  # 1  CAND_0055905   composite=0.8858
       Required=1.000  ExpFit=1.000  Bonus=0.643  Loc=0.490  Avail=0.860  Hire=0.787  Verif=1.000  SoftMult=1.000
       Req signals  : ✓ embeddings  ✓ vectorDB  ✓ python  ✓ eval
       Bonus signals: ✓ finetuning  ✓ L2R  ✗ HR-tech  ✓ distributed  ✗ OSS
       Reasoning: 8.1yr Senior Machine Learning Engineer with evidence of embeddings/retrieval systems, vector DB / hybrid search. bonus: ...

─────────────────────────────────────────────────────────────────
  # 2  CAND_0008425   composite=0.8744
       Required=1.000  ExpFit=1.000  Bonus=0.643  Loc=0.530  Avail=0.849  Hire=0.608  Verif=1.000  SoftMult=1.000
       Req signals  : ✓ embeddings  ✓ vectorDB  ✓ python  ✓ eval
       Bonus signals: ✓ finetuning  ✓ L2R  ✗ HR-tech  ✓ distributed  ✗ OSS
       Reasoning: 7.8yr Senior NLP Engineer with evidence of embeddings/retrieval systems, vector DB / h

## Cell 10 — Run Official Validator & Final Summary

In [10]:
# Run the official validator if available
official_validator = Path("/kaggle/input/redrob-hackathon/validate_submission.py")
if official_validator.exists():
    print("Running official validate_submission.py...")
    import subprocess
    result = subprocess.run(
        [sys.executable, str(official_validator), str(OUTPUT_CSV)],
        capture_output=True, text=True
    )
    print(result.stdout)
    if result.stderr:
        print("STDERR:", result.stderr[:500])
else:
    print("Official validator not found at expected path — inline validation only.")
    print("Run manually: python validate_submission.py submission.csv")

print()
print("=" * 60)
print("  PHASE 2 COMPLETE")
print("=" * 60)
print(f"  Submission: {OUTPUT_CSV}")
print(f"  Top score : {all_results[0]['composite']:.6f}")
print(f"  Rank-10   : {all_results[9]['composite']:.6f}")
print(f"  Rank-100  : {all_results[99]['composite']:.6f}")
print()
print("  Scoring metric: 0.50×NDCG@10 + 0.30×NDCG@50 + 0.15×MAP + 0.05×P@10")
print("  → 80% of score determined by ranks 1-50.")
print("  → Review top-10 in Cell 9 before submitting.")

Official validator not found at expected path — inline validation only.
Run manually: python validate_submission.py submission.csv

  PHASE 2 COMPLETE
  Submission: /content/submission.csv
  Top score : 0.885821
  Rank-10   : 0.810171
  Rank-100  : 0.743108

  Scoring metric: 0.50×NDCG@10 + 0.30×NDCG@50 + 0.15×MAP + 0.05×P@10
  → 80% of score determined by ranks 1-50.
  → Review top-10 in Cell 9 before submitting.
